# Presentations

Moved out of `processing_intra.qmd` - these three cells were made in a rush for a presentation ("radiology day") and need to be sorted out (cleaned up, fixed, or discarded) separately from the main manuscript pipeline.

In [ ]:
# radiology_day_slide1.py

import mindmap as mm
from scripts.data_manager import IntraManager

from mindmap.viz.utils import BaseMesh, BrainMesh, SourceMesh, ChannelMesh, HeadMesh
from mindmap.viz.fancy import create_mesh_animation, create_dynamic_eeg_plot

# data
dm = IntraManager("intra.config.yml")

sub = "48"
channels = ["68", "253", "224", "201", "122"]

src = dm.load(sub, "fwd")["src"]
head = dm.load(sub, "bem")["skin"]
ave = dm.load(sub, "ave", task="spk")
sloreta = dm.load(sub, "stc", task="spk", desc="sloreta")
wcset = dm.load(sub, "stc", task="spk", desc="wcset")


# crop time
tmin = 1.1
tmax = 3.5
ave = ave.crop(tmin=tmin, tmax=tmax)
sloreta = sloreta.crop(tmin=tmin, tmax=tmax)
wcset = wcset.crop(tmin=tmin, tmax=tmax)

# adjust bem scaling
head.points = head.points/1000

# mesh objects
bm_sloreta = SourceMesh(src, sloreta)
bm_wcset = SourceMesh(src, wcset)
cm = ChannelMesh(ave)
hm = HeadMesh(head, line_width=0.01)

# EEG
create_dynamic_eeg_plot(
    ave,
    picks = channels,
    total_duration = 10.0,
    save_path = "eeg.mp4",
)

# sloreta
create_mesh_animation(
    bm_sloreta,
    addons = [cm, hm],
    total_duration = 10.0,
    camera_position = mm.viz.get_brain_view("Lateral Left"),
    save_path = "sloreta.mp4",
)

# wcset
create_mesh_animation(
    bm_wcset,
    addons=[cm, hm],
    total_duration = 10.0,
    camera_position = mm.viz.get_brain_view("Lateral Left"),
    save_path = "wcset.mp4",
)

In [ ]:
# radiology_day_slide2.py

import pyvista as pv

import mindmap as mm
from scripts.data_manager import IntraManager

from mindmap.viz.utils import BaseMesh, BrainMesh, SourceMesh, ChannelMesh, HeadMesh
from mindmap.viz.fancy import create_mesh_animation, create_dynamic_eeg_plot

# data
dm = IntraManager("intra.config.yml")

sub = "48"
channels = ["68", "253", "224", "201", "122"]

src = dm.load(sub, "fwd")["src"]
head = dm.load(sub, "bem")["skin"]
ave = dm.load(sub, "ave", task="spk")
iave = dm.load(sub, "i-ave", task="spk")
sloreta = dm.load(sub, "v-ave", task="spk", desc="sloreta")
dspm = dm.load(sub, "v-ave", task="spk", desc="dspm")
wcset = dm.load(sub, "v-ave", task="spk", desc="wcset")

# adjust bem scaling
head.points = head.points/1000

# standardize
iave = mm.analysis.eeg.normalize_evoked(iave)
sloreta = mm.analysis.eeg.normalize_evoked(sloreta)
dspm = mm.analysis.eeg.normalize_evoked(dspm)
wcset = mm.analysis.eeg.normalize_evoked(wcset)

# HEAD MODEL WITH INTRA ELECTRODES
bm = BrainMesh(src, color="white", opacity=0.3) # add white transparency?
head = HeadMesh(head, line_width=0.01, color="white", style="wireframe")
eeg = ChannelMesh(ave, color="black")
ieeg = ChannelMesh(iave, color="green")

plotter = pv.Plotter()
plotter.add_mesh(bm.hemi_l, **bm.plot_kwargs)
plotter.add_mesh(bm.hemi_r, **bm.plot_kwargs)
plotter.add_mesh(head.polydata, **head.plot_kwargs)
plotter.add_mesh(eeg.polydata, **eeg.plot_kwargs)
plotter.add_mesh(ieeg.polydata, **ieeg.plot_kwargs)
plotter.show()

# I-AVE RECONSTRUCTIONS

# crop time and channels
%matplotlib widget
mm.viz.plot_channel_grid(
    iave,
    {"wcset": wcset, "sloreta": sloreta, "dspm": dspm},
    n_visible = 2,
)

# TEMPORAL CORRELATIONS

In [ ]:
# crop time
tmin, tmax = mm.analysis.eeg.find_spike_window(iave, baseline=(0,1.0), gap_fill_ms=200)
iave = iave.crop(tmin, tmax)
sloreta = sloreta.crop(tmin, tmax)
dspm = dspm.crop(tmin, tmax)
wcset = wcset.crop(tmin, tmax)

metrics = mm.metric.MetricSuite(f"Intracranial EEG, sub-{sub}")

for name, vave in {"sloreta":sloreta, "dspm":dspm, "wcset": wcset}.items():
    corr = mm.analysis.eeg.compute_evoked_correlation(
        iave, vave,
        mode = "temporal",
        method = "log-envelope",
    )

    metrics.register(name, "correlation", corr, chan=iave.ch_names)

target_ch, _ = iave.get_peak(mode="abs")
distances = mm.analysis.eeg.get_channel_distances(iave, target_ch)

# NOTE: cut off here in the original qmd cell - `distances` computed but never used/plotted.